## 1. Setup & Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "torchao>=0.16.0" -q

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizerFast
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import GroupKFold, KFold
from sklearn.isotonic import IsotonicRegression
from scipy.stats import pearsonr
from scipy.optimize import minimize_scalar
import numpy as np
import pandas as pd
import json, os, math
from typing import Sequence, Optional, Dict, Tuple, Any, List
from torch.optim.lr_scheduler import LambdaLR
import time, random

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CFG = {
    'model_name':    'FacebookAI/roberta-large',
    'max_len':       128,
    'batch_size':    4,
    'lora_r':        8,
    'lora_alpha':    32,
    'lora_dropout':  0.1,
    'time_dim':      64,
    'user_emb_dim':  32,
    'hidden_dim':    256,

    'max_hist':      10,

    'decay_rate':    0.1,
    'train_csv':     'https://drive.google.com/uc?id=1ve3Fdkb8kKHbTkc6FNKAH6hLOvme17IP',
    'detail_csv':    'https://drive.google.com/uc?id=1Pkjtv0rPHzJ4kMvPV4uHkDDzY096sHOV',
    'marker_csv':    'https://drive.google.com/uc?id=1pzDb3PhJVqWjL0qw9fN7hz3ysGWMmpCH',
    'test_csv':      'https://drive.google.com/uc?id=1L0nFtxOkMHkC30IuOxhtJWo_SNVbCcIc',
    'template_csv':  'https://drive.google.com/uc?id=1lgjK1GAu2Zj4WjqNGbFUf7lDMEROV0hs',
    'gold_csv':      'https://drive.google.com/uc?id=111iGsXFrSdRlg43YCmJIew1nFcGPaqbB',
    'output_csv':    'subtask2b_predictions_v2.csv',
}

## 2. Data Preparation

In [ ]:
def norm_v(v):    return v / 2.0
def norm_a(a):    return a - 1.0
def denorm_v(v):  return v * 2.0
def denorm_a(a):  return a + 1.0

def norm_disp(x):   return x
def denorm_disp(x): return x

def va_prompt(v_raw, a_raw):
    vd = 'positive' if v_raw > 0.5 else ('neutral' if v_raw > -0.5 else 'negative')
    ad = 'activated' if a_raw > 1.2 else ('calm' if a_raw > 0.4 else 'low energy')
    return f'I am feeling {vd} and {ad}.'

In [ ]:
train_raw  = pd.read_csv(CFG['train_csv'])
detail_raw = pd.read_csv(CFG['detail_csv'])
marker_raw = pd.read_csv(CFG['marker_csv'])
test_raw   = pd.read_csv(CFG['test_csv'])
template   = pd.read_csv(CFG['template_csv'])

for df in [train_raw, detail_raw, marker_raw]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.sort_values(['user_id','timestamp'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    df['valence'] = df['valence'].apply(norm_v)
    df['arousal'] = df['arousal'].apply(norm_a)

test_raw['timestamp_min'] = pd.to_datetime(test_raw['timestamp_min'])
test_raw['timestamp_max'] = pd.to_datetime(test_raw['timestamp_max'])

print(f'Train: {len(train_raw)} rows, {train_raw.user_id.nunique()} users')
print(f'Marker: {len(marker_raw)} rows')
print(f'Test: {len(test_raw)} rows')

In [ ]:
all_users = sorted(set(
    train_raw.user_id.unique().tolist() +
    marker_raw.user_id.unique().tolist()
))
user2idx       = {uid: i+1 for i, uid in enumerate(all_users)}
COLD_IDX       = 0
NUM_USERS      = len(user2idx) + 1

def get_uidx(uid): return user2idx.get(uid, COLD_IDX)

for df in [train_raw, detail_raw, marker_raw]:
    df['user_idx'] = df['user_id'].apply(get_uidx)

print(f'NUM_USERS={NUM_USERS}')

In [ ]:
def compute_obs_stats(va: np.ndarray) -> np.ndarray:
    v, a = va[:, 0], va[:, 1]
    n = len(v)
    idx = np.arange(n, dtype=float)
    trend_v = float(np.polyfit(idx, v, 1)[0]) if n > 2 else 0.0
    trend_a = float(np.polyfit(idx, a, 1)[0]) if n > 2 else 0.0
    return np.array([
        float(v.mean()), float(v.std() + 1e-6),
        float(a.mean()), float(a.std() + 1e-6),
        trend_v, trend_a
    ], dtype=np.float32)


def delta_days(timestamps):
    times = pd.to_datetime(timestamps)
    return (pd.Series(times).diff().dt.total_seconds().fillna(0) / 86400.0).values.astype(np.float32)


def clean_text(text, v_norm, a_norm):
    t = str(text).strip() if isinstance(text, str) else ''
    return t if t else va_prompt(denorm_v(v_norm), denorm_a(a_norm))


def build_train_2b(df: pd.DataFrame, max_hist: int) -> List[Dict]:
    samples = []
    for uid, g in df.groupby('user_id'):
        g = g.sort_values('timestamp').reset_index(drop=True)

        if 'disposition_change_valence' not in g.columns: continue
        dv = g['disposition_change_valence'].dropna()
        da = g['disposition_change_arousal'].dropna()
        if len(dv) == 0: continue

        # Cap ke max_hist (ambil yang terbaru)
        g = g.iloc[-max_hist:].reset_index(drop=True)
        m = len(g)

        texts = [clean_text(g.loc[i,'text'], g.loc[i,'valence'], g.loc[i,'arousal'])
                 for i in range(m)]
        va    = g[['valence','arousal']].values.astype(np.float32)

        samples.append({
            'user_id':   uid,
            'user_idx':  int(g['user_idx'].iloc[0]),
            'texts':     texts,
            'timestamps':g['timestamp'].tolist(),
            'va_seq':    va,
            'obs_stats': compute_obs_stats(va),
            'target_v':  float(dv.iloc[0]),
            'target_a':  float(da.iloc[0]),
            'n_obs':     m,
        })
    return samples


def build_test_2b(marker_df: pd.DataFrame, test_df: pd.DataFrame,
                  max_hist: int) -> List[Dict]:
    samples = []
    for _, row in test_df.iterrows():
        uid  = row['user_id']
        uidx = get_uidx(uid)

        g = marker_df[
            (marker_df['user_id'] == uid) &
            (marker_df['timestamp'] <= row['timestamp_max'])
        ].sort_values('timestamp').reset_index(drop=True)

        if len(g) == 0:
            texts = [va_prompt(0.0, 1.0)]
            va    = np.zeros((1, 2), dtype=np.float32)
        else:
            g = g.iloc[-max_hist:].reset_index(drop=True)
            texts = [clean_text(g.loc[i,'text'], g.loc[i,'valence'], g.loc[i,'arousal'])
                     for i in range(len(g))]
            va    = g[['valence','arousal']].values.astype(np.float32)

        samples.append({
            'user_id':   uid,
            'user_idx':  uidx,
            'texts':     texts,
            'timestamps':g['timestamp'].tolist() if len(g) > 0 else [row['timestamp_max']],
            'va_seq':    va,
            'obs_stats': compute_obs_stats(va),
            'n_obs':     len(va),
        })
    return samples

In [ ]:
full_hist = pd.concat([train_raw, detail_raw]).drop_duplicates(
    subset=['user_id','timestamp']
).sort_values(['user_id','timestamp']).reset_index(drop=True)

train_samples = build_train_2b(full_hist, CFG['max_hist'])
test_samples  = build_test_2b(marker_raw, test_raw, CFG['max_hist'])
print(f'Train samples: {len(train_samples)} | Test samples: {len(test_samples)}')

In [ ]:
class DispDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=128, max_hist=20):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.max_hist  = max_hist

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s  = self.samples[idx]
        m  = len(s['texts'])
        enc = self.tokenizer(
            s['texts'], padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'],         # [m, L]
            'attention_mask': enc['attention_mask'],
            'delta_time':     torch.tensor(delta_days(s['timestamps']), dtype=torch.float),
            'va_seq':         torch.tensor(s['va_seq'], dtype=torch.float),
            'obs_stats':      torch.tensor(s['obs_stats'], dtype=torch.float),
            'user_idx':       torch.tensor(s['user_idx'], dtype=torch.long),
            'n_obs':          torch.tensor(m, dtype=torch.long),
        }
        if 'target_v' in s:
            item['target'] = torch.tensor([s['target_v'], s['target_a']], dtype=torch.float)
        return item

In [ ]:
def collate_variable(batch):
    max_m = max(b['input_ids'].shape[0] for b in batch)
    inp_ids, attn, dt, va = [], [], [], []

    pad_ids  = batch[0]['input_ids'][0:1].clone()
    pad_mask = torch.zeros(1, batch[0]['attention_mask'].shape[1], dtype=torch.long)  # mask=0 → diabaikan

    for b in batch:
        m   = b['input_ids'].shape[0]
        pad = max_m - m
        if pad > 0:
            pad_tile_ids  = pad_ids.expand(pad, -1)
            pad_tile_mask = pad_mask.expand(pad, -1)
            inp_ids.append(torch.cat([b['input_ids'],  pad_tile_ids],  0))
            attn.append(   torch.cat([b['attention_mask'], pad_tile_mask], 0))
            dt.append(     torch.cat([b['delta_time'], torch.zeros(pad)], 0))
            va.append(     torch.cat([b['va_seq'],     torch.zeros(pad, 2)], 0))
        else:
            inp_ids.append(b['input_ids'])
            attn.append(b['attention_mask'])
            dt.append(b['delta_time'])
            va.append(b['va_seq'])

    out = {
        'input_ids':      torch.stack(inp_ids),
        'attention_mask': torch.stack(attn),
        'delta_time':     torch.stack(dt),
        'va_seq':         torch.stack(va),
        'obs_stats':      torch.stack([b['obs_stats'] for b in batch]),
        'user_idx':       torch.stack([b['user_idx']  for b in batch]),
        'n_obs':          torch.stack([b['n_obs']     for b in batch]),
    }
    if all('target' in b for b in batch):
        out['target'] = torch.stack([b['target'] for b in batch])
    return out

## 3. Model Architecture

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(1, dim), nn.ReLU(), nn.Linear(dim, dim))
    def forward(self, dt): return self.mlp(dt.unsqueeze(-1))


class DispForecaster2B(nn.Module):
    """
    Dispositional Change Forecaster
    1. Decayed mean pooling for GRU outputs:
       w_i = exp(-decay_rate * (n - i))
    2. obs_stats concat to head: mean, std, trend V&A 
    3. Direct regression
    """
    def __init__(self, num_users, max_hist=20,
                 user_emb_dim=32, time_dim=64,
                 hidden_dim=256, decay_rate=0.1):
        super().__init__()
        self.decay_rate = decay_rate

        # Backbone
        roberta  = RobertaModel.from_pretrained('FacebookAI/roberta-large')
        lora_cfg = LoraConfig(r=8, lora_alpha=32,
                              target_modules=['query','value'],
                              lora_dropout=0.1, bias='none')
        self.roberta = get_peft_model(roberta, lora_cfg)
        text_dim = self.roberta.config.hidden_size   # 1024

        self.time_emb = TimeEmbedding(time_dim)
        self.user_emb = nn.Embedding(num_users, user_emb_dim, padding_idx=0)

        fused_dim   = text_dim + time_dim + 2 + user_emb_dim
        self.fusion = nn.Linear(fused_dim, hidden_dim)

        # GRU encode
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True,
                          num_layers=2, dropout=0.1)

        # Direct regression head: [hidden + obs_stats(6)] → 2
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + 6, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, 2),
        )

    def encode_texts(self, input_ids, attention_mask):
        B, M, L = input_ids.shape
        out = self.roberta(
            input_ids=input_ids.view(B*M, L),
            attention_mask=attention_mask.view(B*M, L)
        )
        mask = attention_mask.view(B*M, L).unsqueeze(-1).float()

        denom    = mask.sum(1).clamp(min=1)          # [B*M, 1]
        mean_pool = (out.last_hidden_state * mask).sum(1) / denom

        is_padded = (attention_mask.view(B*M, L).sum(-1) == 0)  # [B*M]
        mean_pool[is_padded] = 0.0

        return mean_pool.view(B, M, -1)

    def forward(self, input_ids, attention_mask, delta_time,
                va_seq, user_idx, obs_stats, n_obs):
        B, M = input_ids.shape[:2]

        h_text = self.encode_texts(input_ids, attention_mask)
        h_time = self.time_emb(delta_time)
        h_user = self.user_emb(user_idx).unsqueeze(1).expand(-1, M, -1)

        fused = torch.relu(self.fusion(
            torch.cat([h_text, h_time, va_seq, h_user], dim=-1)
        ))

        gru_out, _ = self.gru(fused)   # [B, M, H]

        # Decayed mean pooling
        # w_i = exp(-decay * (M - i - 1))
        decay_w = torch.zeros(B, M, 1, device=input_ids.device)
        for b in range(B):
            n  = min(int(n_obs[b].item()), M)
            wi = torch.exp(-self.decay_rate *
                           torch.arange(n, 0, -1, dtype=torch.float,
                                        device=input_ids.device))
            decay_w[b, M-n:, 0] = wi / wi.sum()

        seg_repr = (gru_out * decay_w).sum(1)   # [B, H]

        combined = torch.cat([seg_repr, obs_stats], dim=-1)
        return self.head(combined)   # [B, 2]

## 4. Training & Evaluation Setup

In [ ]:
def ccc_loss(pred, target, eps=1e-8):
    if pred.shape[0] < 2:
        return torch.abs(pred - target).mean()
    pm, tm = pred.mean(), target.mean()
    pv = pred.var(unbiased=False)
    tv = target.var(unbiased=False)

    if tv < eps or pv < eps:
        return torch.abs(pred - target).mean()
    cov = ((pred - pm) * (target - tm)).mean()
    ccc = 2.0 * cov / (pv + tv + (pm - tm) ** 2 + eps)
    return 1.0 - ccc

def total_loss(pred, target, w_ccc=0.7, w_mae=0.3):
    lv = w_ccc*ccc_loss(pred[:,0], target[:,0]) + \
         w_mae*torch.abs(pred[:,0]-target[:,0]).mean()
    la = w_ccc*ccc_loss(pred[:,1], target[:,1]) + \
         w_mae*torch.abs(pred[:,1]-target[:,1]).mean()

    loss = lv + la
    if torch.isnan(loss):
        loss = (torch.abs(pred[:,0]-target[:,0]).mean() +
                torch.abs(pred[:,1]-target[:,1]).mean())
    return loss
def _pearson(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 2 or np.var(x) == 0 or np.var(y) == 0: return float('nan'), float('nan')
    return pearsonr(x, y)

def freeze_backbone(model):
    for n, p in model.named_parameters():
        if 'roberta' in n: p.requires_grad = False
    print(f'  frozen. trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def unfreeze_lora(model):
    for n, p in model.named_parameters():
        if 'lora_' in n: p.requires_grad = True
    print(f'  LoRA unfrozen. trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def opt_s1(model):
    return torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.01)

def opt_s2(model):
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (bb if 'roberta' in n else hd).append(p)
    return torch.optim.AdamW([{'params': bb, 'lr': 5e-6}, {'params': hd, 'lr': 1e-4}], weight_decay=0.01)

def warmup_cos(opt, n_warm, n_total):
    def lam(s):
        if s < n_warm: return s/max(n_warm,1)
        return max(0, 0.5*(1+math.cos(math.pi*(s-n_warm)/max(n_total-n_warm,1))))
    return LambdaLR(opt, lam)


def to_dev(batch):
    return {k: batch[k].to(device)
            for k in ['input_ids','attention_mask','delta_time',
                      'va_seq','obs_stats','user_idx','n_obs']}

def train_epoch(model, loader, opt, sch):
    model.train(); tot = 0.0
    for batch in loader:
        opt.zero_grad()
        pred = model(**to_dev(batch))
        loss = total_loss(pred, batch['target'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
        tot += loss.item()
    return tot / max(len(loader), 1)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    pv, pa, tv, ta, uids = [], [], [], [], []
    for batch in loader:
        pred = model(**to_dev(batch)).cpu().numpy()
        tgt  = batch['target'].numpy()
        pv.extend(pred[:,0]); tv.extend(tgt[:,0])
        pa.extend(pred[:,1]); ta.extend(tgt[:,1])
        uids.extend(batch['user_idx'].cpu().tolist())
    r_v, _ = _pearson(pv, tv); r_a, _ = _pearson(pa, ta)
    mae_v  = float(np.nanmean(np.abs(np.array(pv)-np.array(tv))))
    mae_a  = float(np.nanmean(np.abs(np.array(pa)-np.array(ta))))
    print(f'    V r={r_v:.4f} mae={mae_v:.4f} | A r={r_a:.4f} mae={mae_a:.4f}')
    return {'v_r': r_v, 'a_r': r_a, 'pv': pv, 'pa': pa}

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(CFG['model_name'])

def make_model():
    return DispForecaster2B(
        num_users=NUM_USERS, max_hist=CFG['max_hist'],
        user_emb_dim=CFG['user_emb_dim'], time_dim=CFG['time_dim'],
        hidden_dim=CFG['hidden_dim'], decay_rate=CFG['decay_rate'])

gold_df = pd.read_csv(CFG['gold_csv'])
print('tokenizer + make_model + gold_df siap')

In [ ]:
RESULTS_DIR = '/content/drive/MyDrive/Analysis Subtask 2b/1'
for sub in ['ckpt','preds','analysis']:
    os.makedirs(f'{RESULTS_DIR}/{sub}', exist_ok=True)
print('results:', RESULTS_DIR)

In [ ]:
def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
def _pearson_loss(pred, target):
    def r1(p, t):
        if p.shape[0] < 2: return torch.abs(p - t).mean()
        pm, tm = p.mean(), t.mean(); pv = p.var(unbiased=False); tv = t.var(unbiased=False)
        if pv < 1e-8 or tv < 1e-8: return torch.abs(p - t).mean()
        cov = ((p - pm) * (t - tm)).mean()
        return 1.0 - cov / torch.sqrt(pv * tv + 1e-8)
    return r1(pred[:,0], target[:,0]) + r1(pred[:,1], target[:,1])

def make_loss(kind):
    if kind == 'ccc':     return total_loss
    if kind == 'mae':     return lambda p, t: torch.abs(p - t).mean()
    if kind == 'pearson': return _pearson_loss
    raise ValueError(kind)

def train_epoch_v(model, loader, opt, sch, loss_fn):
    model.train(); tot = 0.0
    for batch in loader:
        opt.zero_grad()
        pred = model(**to_dev(batch))
        loss = loss_fn(pred, batch['target'].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step(); tot += loss.item()
    return tot / max(len(loader), 1)

## 5. Experiment

In [ ]:
VARIANTS = [
    {'name':'RoBERTa_GRU_ccc', 'loss':'ccc', 'make': make_model},
]
SEED = 42
S1, S2 = 3, 7

In [ ]:
def train_variant(variant):
    set_seed(SEED)
    loss_fn = make_loss(variant['loss'])
    samples_df = pd.DataFrame([{'user_id': s['user_id'], 'idx': i} for i, s in enumerate(train_samples)])
    gkf = GroupKFold(n_splits=5)
    bsv, bv = None, -1e9; bsa, ba = None, -1e9
    for fold,(tr_idx,va_idx) in enumerate(gkf.split(samples_df, groups=samples_df['user_id'])):
        tr_s = [train_samples[i] for i in samples_df.iloc[tr_idx]['idx']]
        va_s = [train_samples[i] for i in samples_df.iloc[va_idx]['idx']]
        tr_dl = DataLoader(DispDataset(tr_s, tokenizer, CFG['max_len'], CFG['max_hist']),
                           batch_size=CFG['batch_size'], shuffle=True, drop_last=True, collate_fn=collate_variable)
        va_dl = DataLoader(DispDataset(va_s, tokenizer, CFG['max_len'], CFG['max_hist']),
                           batch_size=CFG['batch_size'], shuffle=False, collate_fn=collate_variable)
        model = variant['make']()
        model.to(device) # Move the model to the GPU
        freeze_backbone(model); opt=opt_s1(model); sch=warmup_cos(opt,len(tr_dl),len(tr_dl)*S1)
        for ep in range(S1):
            train_epoch_v(model,tr_dl,opt,sch,loss_fn); m=validate(model,va_dl)
            if not np.isnan(m['v_r']) and m['v_r']>bv: bv=m['v_r']; bsv={k:v.cpu().clone() for k,v in model.state_dict().items()}
            if not np.isnan(m['a_r']) and m['a_r']>ba: ba=m['a_r']; bsa={k:v.cpu().clone() for k,v in model.state_dict().items()}
        unfreeze_lora(model); opt=opt_s2(model); sch=warmup_cos(opt,len(tr_dl),len(tr_dl)*S2)
        for ep in range(S2):
            train_epoch_v(model,tr_dl,opt,sch,loss_fn); m=validate(model,va_dl)
            if not np.isnan(m['v_r']) and m['v_r']>bv: bv=m['v_r']; bsv={k:v.cpu().clone() for k,v in model.state_dict().items()}
            if not np.isnan(m['a_r']) and m['a_r']>ba: ba=m['a_r']; bsa={k:v.cpu().clone() for k,v in model.state_dict().items()}
        del model; torch.cuda.empty_cache()
    torch.save(bsv, f"{RESULTS_DIR}/ckpt/2b_{variant['name']}_v.pt")
    torch.save(bsa, f"{RESULTS_DIR}/ckpt/2b_{variant['name']}_a.pt")
    return bsv, bsa

In [ ]:
@torch.no_grad()
def predict_aligned(bsv, bsa, variant):
    mv = variant['make']().to(device); mv.load_state_dict(bsv); mv.eval()
    ma = variant['make']().to(device); ma.load_state_dict(bsa); ma.eval()
    ds = DispDataset(test_samples, tokenizer, CFG['max_len'], CFG['max_hist'])
    loader = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=False, collate_fn=collate_variable)
    rv, ra = [], []
    for batch in loader:
        b = to_dev(batch)
        rv += mv(**b)[:,0].cpu().numpy().tolist()
        ra += ma(**b)[:,1].cpu().numpy().tolist()
    del mv, ma; torch.cuda.empty_cache()
    return pd.DataFrame({'user_id':[s['user_id'] for s in test_samples],'pred_v':rv,'pred_a':ra})

def kfold_isotonic(pred, gold, n=5, seed=42):
    pred=np.asarray(pred,float); gold=np.asarray(gold,float)
    if np.var(pred)==0 or len(pred)<n+1: return float('nan')
    oof=np.full_like(pred,np.nan)
    for tr,te in KFold(min(n,len(pred)),shuffle=True,random_state=seed).split(pred):
        if np.var(pred[tr])==0: continue
        oof[te]=IsotonicRegression(out_of_bounds='clip').fit(pred[tr],gold[tr]).predict(pred[te])
    m=~np.isnan(oof)
    return float(pearsonr(oof[m],gold[m])[0]) if m.sum()>2 and np.var(oof[m])>0 else float('nan')

In [ ]:
ABL_CSV = f'{RESULTS_DIR}/analysis/variants_2b.csv'

In [ ]:
def compare_variants():
    if not os.path.exists(ABL_CSV): print('belum ada hasil.'); return
    df = pd.read_csv(ABL_CSV).sort_values('avg', ascending=False)
    print('r mentah vs post-process (isotonic K-fold) + verdict diagnosa per varian:')
    print(df[['name','loss','r_v','r_a','avg','r_v_pp','r_a_pp','verdict']].to_string(index=False))
    todo = [v['name'] for v in VARIANTS if v['name'] not in set(df['name'])]
    if todo: print(f'\nbelum dijalankan: {todo} — panggil run_one_variant() lagi')
    return df

In [ ]:
def _done():
    if not os.path.exists(ABL_CSV): return set()
    return set(pd.read_csv(ABL_CSV)['name'].tolist())

def run_one_variant(only=None):
    done = _done()
    v = next((x for x in VARIANTS if x['name']==only), None) if only \
        else next((x for x in VARIANTS if x['name'] not in done), None)
    if v is None:
        print('semua varian selesai.'); return compare_variants()
    print(f">>> VARIAN: {v['name']} (loss={v['loss']}) — training...")
    t0=time.time()
    bsv, bsa = train_variant(v)
    pred_df = predict_aligned(bsv, bsa, v)
    pred_df.to_csv(f"{RESULTS_DIR}/preds/2b_{v['name']}_raw.csv", index=False)
    m = pd.merge(pred_df, gold_df[['user_id','disp_change_valence','disp_change_arousal']].rename(
        columns={'disp_change_valence':'gold_v','disp_change_arousal':'gold_a'}), on='user_id')
    rv = pearsonr(m['pred_v'],m['gold_v'])[0] if m['pred_v'].var()>0 else float('nan')
    ra = pearsonr(m['pred_a'],m['gold_a'])[0] if m['pred_a'].var()>0 else float('nan')
    rv_pp = kfold_isotonic(m['pred_v'],m['gold_v']); ra_pp = kfold_isotonic(m['pred_a'],m['gold_a'])
    print(f"\n--- DIAGNOSA {v['name']} ---")
    verdict = diagnose_2b(pred_df, gold_df)
    row = {'name':v['name'],'loss':v['loss'],'r_v':round(rv,4),'r_a':round(ra,4),
           'r_v_pp':round(rv_pp,4),'r_a_pp':round(ra_pp,4),
           'avg':round(np.nanmean([rv,ra]),4),'verdict':verdict,'minutes':round((time.time()-t0)/60,1)}
    pd.DataFrame([row]).to_csv(ABL_CSV, mode='a', header=not os.path.exists(ABL_CSV), index=False)
    print(f"\n=== {v['name']} selesai: avg r={row['avg']} verdict={verdict} ===")

In [ ]:

compare_variants()
run_one_variant()